In [2]:
import torch
import wandb
from torch.optim.lr_scheduler import CosineAnnealingLR
from training import train_epoch, eval_epoch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune

from lth_utils import get_prunable_layers, train_model, remove_pruning_reparam, count_sparsity, count_active_parameters

from models import TeacherCNN, StudentCNN, BigCNN

In [3]:
from mnist1d_dataset import get_mnist1d_datasets
from torch.utils.data import DataLoader
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds, test_ds = get_mnist1d_datasets()

train_loader = DataLoader(
    train_ds,
    batch_size=256,
    shuffle=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=256,
    shuffle=False
)

File already exists. Skipping download.
Successfully loaded data from ./mnist1d_data.pkl


## KD Training

In [6]:
wandb.init(
    project="mnist-1d-KDvsLTH",
    group="TeacherCNN",
    name="TeacherCNN",
    config={
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "epochs": 500,
        "patience": 50,
        "architecture": "TeacherCNN",
        "scheduler": "CosineAnnealingLR"
    }
)

teacher = TeacherCNN().to(device)
optimizer = torch.optim.Adam(teacher.parameters(), lr=wandb.config.learning_rate, weight_decay=wandb.config.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=wandb.config.epochs)
criterion = nn.CrossEntropyLoss()
epochs = wandb.config.epochs
#early_stopper = EarlyStopping(patience=wandb.config.patience)

wandb.watch(teacher, log_freq=100)

best_test_acc = 0.0
model_path = "models/best_teacher_model.pth"

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(teacher, train_loader, optimizer, criterion, device)
    test_loss, test_acc = eval_epoch(teacher, test_loader, criterion, device)

    # Step the scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    wandb.log({
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "learning_rate": current_lr
    }, step=epoch)

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(teacher.state_dict(), model_path)
        wandb.save(model_path)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:03d} | Acc: {test_acc:.3f} | LR: {current_lr:.6f}")

print("\nTraining complete.")

teacher.load_state_dict(torch.load(model_path))

final_loss, final_acc = eval_epoch(teacher, test_loader, nn.CrossEntropyLoss(), device)

wandb.log({
    "best_test_loss": final_loss,
    "best_test_acc": final_acc
})

print(f"Best Teacher model - Acc: {final_acc:.4f} | Loss: {final_loss:.4f}")

wandb.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


learning_rate,▁
test_acc,▁
test_loss,▁
train_acc,▁
train_loss,▁
learning_rate,0.001
test_acc,0.498
test_loss,1.36612
train_acc,0.2905
train_loss,1.79066


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch 010 | Acc: 0.936 | LR: 0.000999
Epoch 020 | Acc: 0.977 | LR: 0.000996
Epoch 030 | Acc: 0.977 | LR: 0.000991
Epoch 040 | Acc: 0.981 | LR: 0.000984
Epoch 050 | Acc: 0.985 | LR: 0.000976
Epoch 060 | Acc: 0.978 | LR: 0.000965
Epoch 070 | Acc: 0.978 | LR: 0.000952
Epoch 080 | Acc: 0.983 | LR: 0.000938
Epoch 090 | Acc: 0.985 | LR: 0.000922
Epoch 100 | Acc: 0.983 | LR: 0.000905
Epoch 110 | Acc: 0.985 | LR: 0.000885
Epoch 120 | Acc: 0.982 | LR: 0.000864
Epoch 130 | Acc: 0.985 | LR: 0.000842
Epoch 140 | Acc: 0.983 | LR: 0.000819
Epoch 150 | Acc: 0.982 | LR: 0.000794
Epoch 160 | Acc: 0.987 | LR: 0.000768
Epoch 170 | Acc: 0.987 | LR: 0.000741
Epoch 180 | Acc: 0.981 | LR: 0.000713
Epoch 190 | Acc: 0.983 | LR: 0.000684
Epoch 200 | Acc: 0.983 | LR: 0.000655
Epoch 210 | Acc: 0.984 | LR: 0.000624
Epoch 220 | Acc: 0.987 | LR: 0.000594
Epoch 230 | Acc: 0.988 | LR: 0.000563
Epoch 240 | Acc: 0.986 | LR: 0.000531
Epoch 250 | Acc: 0.987 | LR: 0.000500
Epoch 260 | Acc: 0.988 | LR: 0.000469
Epoch 270 | 

best_test_acc,▁
best_test_loss,▁
learning_rate,█████▇▇▇▇▇▇▇▆▆▆▅▅▅▅▅▄▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
test_acc,▁███████████████████████████████████████
test_loss,▇▅█▄▆▅▅▅▄▃▆▅▃▄▆▅▅▅▄▄▁▂▂▃▂▂▂▂▁▂▁▁▂▁▂▁▂▂▁▁
train_acc,▁▅█▇████████████████████████████████████
train_loss,█▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_test_acc,0.994
best_test_loss,0.02109
learning_rate,0
test_acc,0.99


### Distillation Training functions

In [7]:
import torch.nn.functional as F

def distillation_loss(student_logits, teacher_logits, targets, T=4.0, alpha=0.3):
    # Hard loss
    ce = F.cross_entropy(student_logits, targets)

    # Soft loss
    p_s = F.log_softmax(student_logits / T, dim=1)
    p_t = F.softmax(teacher_logits / T, dim=1)

    kl = F.kl_div(p_s, p_t, reduction="batchmean")

    return alpha * ce + (1 - alpha) * (T ** 2) * kl


In [8]:
def train_kd_epoch(student, teacher, loader, optimizer, T=4.0, alpha=0.3, device=torch.device("cpu")):
    student.train()
    total_loss = 0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        with torch.no_grad():
            teacher_logits = teacher(x)

        student_logits = student(x)
        loss = distillation_loss(
            student_logits,
            teacher_logits,
            y,
            T=T,
            alpha=alpha
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

        # accuracy
        preds = student_logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    avg_loss = total_loss / total
    acc = correct / total

    return avg_loss, acc


In [9]:
@torch.no_grad()
def eval_kd_epoch(student, teacher, loader, T=4.0, alpha=0.3, device=torch.device("cpu")):
    """
    Evaluate the student on a dataset.
    If teacher is provided, compute distillation loss.
    Returns average loss and accuracy.
    """
    student.eval()
    total_loss = 0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        student_logits = student(x)

        if teacher is not None:
            teacher_logits = teacher(x)
            loss = distillation_loss(student_logits, teacher_logits, y, T=T, alpha=alpha)
        else:
            loss = F.cross_entropy(student_logits, y)

        total_loss += loss.item() * x.size(0)
        preds = student_logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc


### Knoledge Distillation

In [10]:
teacher = TeacherCNN().to(device)
teacher.load_state_dict(torch.load("models/best_teacher_model.pth", map_location=device))

n_students = 10
base_seed = 42

for student_id in range(n_students):

    run = wandb.init(
        project="mnist-1d-KDvsLTH",
        name=f"KD-Student-{student_id}",
        group="StudentCNN",
        reinit=True,
        config={
            "student_id": student_id,
            "learning_rate": 1e-2,
            "weight_decay": 1e-4,
            "T_start": 12,
            "T_end": 8,
            "alpha": 0.2,
            "epochs": 200,
            "architecture": "StudentCNN",
            "scheduler": "CosineAnnealingLR",
            "train_seed": base_seed + student_id,
        }
    )

    # ---- different generator per student ----
    g = torch.Generator()
    g.manual_seed(base_seed + student_id)

    train_loader = DataLoader(
        train_ds,
        batch_size=128,
        shuffle=True,
        generator=g
    )

    # ---- model + optimizer ----
    student = StudentCNN().to(device)
    optimizer = torch.optim.Adam(
        student.parameters(),
        lr=wandb.config.learning_rate,
        weight_decay=wandb.config.weight_decay
    )

    scheduler = CosineAnnealingLR(optimizer, T_max=wandb.config.epochs)

    best_test_acc = 0.0
    start_time = time.time()

    try:
        for epoch in range(wandb.config.epochs):

            # temperature schedule
            T = (
                wandb.config.T_start
                - (wandb.config.T_start - wandb.config.T_end)
                * (epoch / (wandb.config.epochs - 1))
            )

            train_loss, train_acc = train_kd_epoch(
                student, teacher, train_loader, optimizer,
                T=T, alpha=wandb.config.alpha, device=device
            )

            test_loss, test_acc = eval_kd_epoch(
                student, teacher, test_loader,
                T=T, alpha=wandb.config.alpha, device=device
            )

            scheduler.step()

            wandb.log({
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "test_loss": test_loss,
                "test_acc": test_acc,
                "temperature": T,
            })

            if test_acc > best_test_acc:
                best_test_acc = test_acc
                ckpt_name = f"models/best_kd_student_{student_id}.pth"
                torch.save(student.state_dict(), ckpt_name)
                wandb.save(ckpt_name)

    finally:
        training_time = time.time() - start_time

        student.load_state_dict(
            torch.load(f"models/best_kd_student_{student_id}.pth")
        )

        final_loss, final_acc = eval_epoch(
            student, test_loader, nn.CrossEntropyLoss(), device
        )

        wandb.log({
            "final_best_test_loss": final_loss,
            "final_best_test_acc": final_acc,
            "total_training_time_sec": training_time,
        })

        wandb.finish()


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇█
final_best_test_acc,▁
final_best_test_loss,▁
temperature,█████▇▇▆▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁
test_acc,▁▅▇▇███▇███▇████████████████████████████
test_loss,█▅▆▆▄▄▄▄▅▄▄▃▄▄▃▃▃▂▃▃▂▃▂▂▂▂▂▂▁▂▂▂▂▁▂▂▁▁▁▁
total_training_time_sec,▁
train_acc,▁▃▆▇▇▇▇▇▇▇██████████████████████████████
train_loss,█▇▅▅▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,199
final_best_test_acc,0.972


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇██
final_best_test_acc,▁
final_best_test_loss,▁
temperature,███▇▇▇▇▆▆▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▁
test_acc,▁▇▇▇▇▇██████████████████████████████████
test_loss,█▆▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_training_time_sec,▁
train_acc,▁▃▄▅▆▇▇▇▇███████████████████████████████
train_loss,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,199
final_best_test_acc,0.971


...

epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
final_best_test_acc,▁
final_best_test_loss,▁
temperature,███▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁
test_acc,▁▄▇▇▇███████████████████████████████████
test_loss,█▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_training_time_sec,▁
train_acc,▁▄▆▇████████████████████████████████████
train_loss,█▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,199
final_best_test_acc,0.963


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▁▂▂▂▂▂▂▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇███████
final_best_test_acc,▁
final_best_test_loss,▁
temperature,███▇▇▇▇▇▆▆▆▆▅▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
test_acc,▁▃▄▇████████████████████████████████████
test_loss,█▂▂▂▂▂▂▂▂▂▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_training_time_sec,▁
train_acc,▁▇██████████████████████████████████████
train_loss,█▆▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,199
final_best_test_acc,0.976


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇██████
final_best_test_acc,▁
final_best_test_loss,▁
temperature,███▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁
test_acc,▁▇▇▇████████████████████████████████████
test_loss,█▄▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_training_time_sec,▁
train_acc,▁▅▇█████████████████████████████████████
train_loss,█▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,199
final_best_test_acc,0.965


## LTH Training

In [6]:
import torch.nn.utils.prune as prune
import copy
def run_lth_from_starting_model(model,initial_state, train_loader, test_loader, rounds=5, prune_amount=0.2, epochs_per_round=5,device="cpu"):
    """Recives a trained model to start LTH from, first it prunes it, then rewinds to initial weights."""

    # Set initial weights
    with torch.no_grad():
        for name, param in model.named_parameters():
            clean_name = name.replace("_orig", "")
            if clean_name in initial_state:
                 param.data.copy_(initial_state[clean_name].data)
    
    
    print(f"Initial parameters: {sum(p.numel() for p in model.parameters())}")

    for round_idx in range(rounds):
        print(f"\n--- Round {round_idx} ---")
        
        # Train (Re-init optimizer to reset momentum)
        optimizer = optim.Adam(model.parameters(), lr=0.001) 
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs_per_round)
        
        history = train_model(
        model, 
        device, 
        train_loader, 
        test_loader, 
        optimizer,
        scheduler,
        epochs=epochs_per_round,
        round_idx=round_idx 
        )

        best_acc = max(history['val_acc'])

        # Calculate Sparsity
        sparsity_pct = count_sparsity(model)
        print(f"Sparsity after pruning: {sparsity_pct:.2f}%")
        active_params = count_active_parameters(model)
        print(f"Active parameters after pruning: {active_params}")

        # Prune
        parameters_to_prune = get_prunable_layers(model)
        prune.global_unstructured(
            parameters_to_prune,
            pruning_method=prune.L1Unstructured,
            amount=prune_amount,
        )

        wandb.log({
            "active_parameters": active_params,
            "sparsity_percentage": sparsity_pct,
            "round": round_idx,
            "best_val_acc": best_acc,
            })
        
        # Reset Weights (The Rewind)
        with torch.no_grad():
            for name, param in model.named_parameters():
                clean_name = name.replace("_orig", "")
                if clean_name in initial_state:
                     param.data.copy_(initial_state[clean_name].data)

        print("Weights reset to initialization.")

    remove_pruning_reparam(model)

    return model

### Lottery Tickets

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

rounds = 9
num_tickets = 10
base_seed = 42

dense_model = BigCNN().to(device)
initial_state = copy.deepcopy(dense_model.state_dict())

for run_idx in range(num_tickets):

    seed = base_seed + run_idx * 100  # spaced seeds
    g = torch.Generator().manual_seed(seed)

    train_loader = DataLoader(
        train_ds,
        batch_size=128,
        shuffle=True,
        generator=g
    )

    # Fresh model EVERY time
    model = BigCNN().to(device)
    

    wandb.init(
        project="mnist-1d-KDvsLTH",
        name=f"LTH-CNN-Run{run_idx}",
        group="Linear-mode-connectivity",
        config={
            "learning_rate": 0.001,
            "rounds": rounds,
            "prune_amount": 0.2,
            "epochs_per_round": 200,
            "architecture": "BigCNN",
            "seed": seed
        }
    )

    _ = run_lth_from_starting_model(
        model,
        initial_state,
        train_loader,
        test_loader,
        rounds=rounds,
        prune_amount=0.2,
        epochs_per_round=200,
        device=device
    )

    # Save dense model (aliasing is fine)
    torch.save(
        torch.load(f"best_lth_model_round_{0}.pth"),
        f"models/lth/dense_model_run{run_idx}.pth"
    )
    # Save final ticket (aliasing is fine)
    torch.save(
        torch.load(f"best_lth_model_round_{rounds-1}.pth"),
        f"models/lth/ticket_model_run{run_idx}.pth"
    )

    wandb.finish()


Initial parameters: 60938

--- Round 0 ---
Round 0 | Epoch 1 | Val Acc: 37.00%
Round 0 | Epoch 2 | Val Acc: 61.40%
Round 0 | Epoch 3 | Val Acc: 68.90%
Round 0 | Epoch 4 | Val Acc: 71.90%
Round 0 | Epoch 5 | Val Acc: 73.10%
Round 0 | Epoch 6 | Val Acc: 72.70%
Round 0 | Epoch 7 | Val Acc: 72.80%
Round 0 | Epoch 8 | Val Acc: 73.20%
Round 0 | Epoch 9 | Val Acc: 75.40%
Round 0 | Epoch 10 | Val Acc: 75.90%
Round 0 | Epoch 11 | Val Acc: 80.40%
Round 0 | Epoch 12 | Val Acc: 80.00%
Round 0 | Epoch 13 | Val Acc: 84.20%
Round 0 | Epoch 14 | Val Acc: 85.70%
Round 0 | Epoch 15 | Val Acc: 87.30%
Round 0 | Epoch 16 | Val Acc: 89.80%
Round 0 | Epoch 17 | Val Acc: 91.20%
Round 0 | Epoch 18 | Val Acc: 90.80%
Round 0 | Epoch 19 | Val Acc: 91.30%
Round 0 | Epoch 20 | Val Acc: 91.10%
Round 0 | Epoch 21 | Val Acc: 92.00%
Round 0 | Epoch 22 | Val Acc: 91.70%
Round 0 | Epoch 23 | Val Acc: 92.30%
Round 0 | Epoch 24 | Val Acc: 92.60%
Round 0 | Epoch 25 | Val Acc: 94.90%
Round 0 | Epoch 26 | Val Acc: 94.10%
Roun

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Round 8 | Epoch 200 | Val Acc: 98.20%
Sparsity after pruning: 83.22%
Layer: features.0           | Active:      126 /      160 (78.75%)
Layer: features.5           | Active:     1239 /    10240 (12.10%)
Layer: features.10          | Active:     3309 /    40960 (8.08%)
Layer: classifier.1         | Active:     4907 /     8192 (59.90%)
Layer: classifier.4         | Active:      517 /      640 (80.78%)
------------------------------------------------------------
TOTAL ACTIVE PARAMETERS: 10,098 / 60,192
GLOBAL DENSITY: 16.78%
Active parameters after pruning: 10098
Weights reset to initialization.


active_parameters,█▆▅▄▃▂▂▁▁
best_val_acc,▃▆▇▄▇█▆▂▁
round,▁▂▃▄▅▅▆▇█
round_0/train_loss,█▇▅▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
round_0/val_acc,▁▃▅▆▇▇▇▇████████████████████████████████
round_1/train_loss,██▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
round_1/val_acc,▁▁▂▇▇▇▇▇▇▇██████████████████████████████
round_2/train_loss,█▇▆▆▆▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
round_2/val_acc,▁▆▆▇▇▇█▇▇▇████████▇█████████████████████
round_3/train_loss,█▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+12,...


Initial parameters: 60938

--- Round 0 ---
Round 0 | Epoch 1 | Val Acc: 35.20%
Round 0 | Epoch 2 | Val Acc: 60.70%
Round 0 | Epoch 3 | Val Acc: 67.40%
Round 0 | Epoch 4 | Val Acc: 74.10%
Round 0 | Epoch 5 | Val Acc: 74.00%
Round 0 | Epoch 6 | Val Acc: 72.00%
Round 0 | Epoch 7 | Val Acc: 72.60%
Round 0 | Epoch 8 | Val Acc: 74.20%
Round 0 | Epoch 9 | Val Acc: 73.10%
Round 0 | Epoch 10 | Val Acc: 75.00%
Round 0 | Epoch 11 | Val Acc: 78.10%
Round 0 | Epoch 12 | Val Acc: 82.90%
Round 0 | Epoch 13 | Val Acc: 84.80%
Round 0 | Epoch 14 | Val Acc: 86.40%
Round 0 | Epoch 15 | Val Acc: 91.60%
Round 0 | Epoch 16 | Val Acc: 92.00%
Round 0 | Epoch 17 | Val Acc: 91.60%
Round 0 | Epoch 18 | Val Acc: 92.80%
Round 0 | Epoch 19 | Val Acc: 93.10%
Round 0 | Epoch 20 | Val Acc: 92.80%
Round 0 | Epoch 21 | Val Acc: 92.40%
Round 0 | Epoch 22 | Val Acc: 94.00%
Round 0 | Epoch 23 | Val Acc: 93.20%
Round 0 | Epoch 24 | Val Acc: 93.40%
Round 0 | Epoch 25 | Val Acc: 94.10%
Round 0 | Epoch 26 | Val Acc: 95.30%
Roun

...

active_parameters,█▆▅▄▃▂▂▁▁
best_val_acc,▇█▅▇▄▅▅▇▁
round,▁▂▃▄▅▅▆▇█
round_0/train_loss,█▄▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
round_0/val_acc,▁▂▃▄▅▆▇▇▇▇▇▇▇▇▇█▇▇▇▇▇███▇▇▇▇▇▇▇▇█▇▇██▇██
round_1/train_loss,███▇▇▄▄▄▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
round_1/val_acc,▁▃▃▄▃▅▇▇▇█▇█████████████████████████████
round_2/train_loss,█▆▅▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
round_2/val_acc,▁▄▇▇▇███████████████████████████████████
round_3/train_loss,█▇▄▄▃▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+12,...


Initial parameters: 60938

--- Round 0 ---
Round 0 | Epoch 1 | Val Acc: 31.80%
Round 0 | Epoch 2 | Val Acc: 62.40%
Round 0 | Epoch 3 | Val Acc: 67.70%
Round 0 | Epoch 4 | Val Acc: 70.70%
Round 0 | Epoch 5 | Val Acc: 71.20%
Round 0 | Epoch 6 | Val Acc: 73.30%
Round 0 | Epoch 7 | Val Acc: 72.80%
Round 0 | Epoch 8 | Val Acc: 73.20%
Round 0 | Epoch 9 | Val Acc: 74.10%
Round 0 | Epoch 10 | Val Acc: 69.40%
Round 0 | Epoch 11 | Val Acc: 73.70%
Round 0 | Epoch 12 | Val Acc: 79.10%
Round 0 | Epoch 13 | Val Acc: 85.10%
Round 0 | Epoch 14 | Val Acc: 86.30%
Round 0 | Epoch 15 | Val Acc: 87.30%
Round 0 | Epoch 16 | Val Acc: 91.00%
Round 0 | Epoch 17 | Val Acc: 91.80%
Round 0 | Epoch 18 | Val Acc: 92.20%
Round 0 | Epoch 19 | Val Acc: 92.40%
Round 0 | Epoch 20 | Val Acc: 91.70%
Round 0 | Epoch 21 | Val Acc: 92.30%
Round 0 | Epoch 22 | Val Acc: 92.20%
Round 0 | Epoch 23 | Val Acc: 92.90%
Round 0 | Epoch 24 | Val Acc: 92.70%
Round 0 | Epoch 25 | Val Acc: 94.20%
Round 0 | Epoch 26 | Val Acc: 93.60%
Roun

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Round 8 | Epoch 200 | Val Acc: 98.60%
Sparsity after pruning: 83.22%
Layer: features.0           | Active:      128 /      160 (80.00%)
Layer: features.5           | Active:     1247 /    10240 (12.18%)
Layer: features.10          | Active:     3285 /    40960 (8.02%)
Layer: classifier.1         | Active:     4913 /     8192 (59.97%)
Layer: classifier.4         | Active:      525 /      640 (82.03%)
------------------------------------------------------------
TOTAL ACTIVE PARAMETERS: 10,098 / 60,192
GLOBAL DENSITY: 16.78%
Active parameters after pruning: 10098
Weights reset to initialization.


active_parameters,█▆▅▄▃▂▂▁▁
best_val_acc,██▇▆▇▇▄▆▁
round,▁▂▃▄▅▅▆▇█
round_0/train_loss,█▆▄▄▂▂▂▂▂▂▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
round_0/val_acc,▁▆▇▇▇▇▇▇▇▇██████████████████████████████
round_1/train_loss,█▇▆▅▅▃▃▃▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
round_1/val_acc,▁▆▇▇▇█▇█████████████████████████████████
round_2/train_loss,█▅▅▅▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
round_2/val_acc,▁▅▆▇▇███████████████████████████████████
round_3/train_loss,█▅▅▄▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+12,...


### Matching models

In [10]:
import copy
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds, test_ds = get_mnist1d_datasets()

test_loader = DataLoader(
    test_ds,
    batch_size=256,
    shuffle=False
)
train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(42)
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
k = 5
model = BigCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
scheduler = CosineAnnealingLR(optimizer, T_max=k)
criterion = nn.CrossEntropyLoss()
for epoch in range(k):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device=device)
        test_loss, test_acc = eval_epoch(model, test_loader, criterion, device=device)
        scheduler.step()
        print(f"Epoch {epoch+1} | Test Acc: {test_acc:.3f} | Train Acc: {train_acc:.3f}")
initial_state = copy.deepcopy(model.state_dict())

rounds = 10
num_matching_models = 2
base_seed = 42

prune_amount = 0.2

for run_idx in range(num_matching_models):

    seed = base_seed + run_idx * 100  # spaced seeds
    g = torch.Generator().manual_seed(seed)

    train_loader = DataLoader(
        train_ds,
        batch_size=128,
        shuffle=True,
        generator=g
    )

    wandb.init(
        project="mnist-1d-KDvsLTH",
        name=f"LTH-CNN-Run{run_idx}",
        group="Linear-mode-connectivity",
        config={
            "learning_rate": 0.001,
            "rounds": rounds,
            "prune_amount": prune_amount,
            "epochs_per_round": 200,
            "architecture": "BigCNN",
            "seed": seed
        }
    )

    model = BigCNN().to(device)

    _ = run_lth_from_starting_model(
        model,
        initial_state,          # IMPORTANT: need to run the dense model cell first
        train_loader,
        test_loader,
        rounds=rounds,
        prune_amount=wandb.config.prune_amount,
        epochs_per_round=wandb.config.epochs_per_round,
        device=device
    )

    # Save dense model (aliasing is fine)
    torch.save(
        torch.load(f"best_lth_model_round_0.pth"),
        f"models/matching/dense_model_run{run_idx}.pth"
    )
    # Save final ticket (aliasing is fine)
    torch.save(
        torch.load(f"best_lth_model_round_{rounds-1}.pth"),
        f"models/matching/matching_model_run{run_idx}.pth"
    )

    wandb.finish()


File already exists. Skipping download.
Successfully loaded data from ./mnist1d_data.pkl
Epoch 1 | Test Acc: 0.591 | Train Acc: 0.462
Epoch 2 | Test Acc: 0.780 | Train Acc: 0.666
Epoch 3 | Test Acc: 0.763 | Train Acc: 0.747
Epoch 4 | Test Acc: 0.868 | Train Acc: 0.814
Epoch 5 | Test Acc: 0.900 | Train Acc: 0.847


Initial parameters: 60938

--- Round 0 ---
Round 0 | Epoch 1 | Val Acc: 89.10%
Round 0 | Epoch 2 | Val Acc: 91.00%
Round 0 | Epoch 3 | Val Acc: 92.10%
Round 0 | Epoch 4 | Val Acc: 92.90%
Round 0 | Epoch 5 | Val Acc: 92.50%
Round 0 | Epoch 6 | Val Acc: 92.50%
Round 0 | Epoch 7 | Val Acc: 92.70%
Round 0 | Epoch 8 | Val Acc: 92.90%
Round 0 | Epoch 9 | Val Acc: 92.90%
Round 0 | Epoch 10 | Val Acc: 93.70%
Round 0 | Epoch 11 | Val Acc: 94.10%
Round 0 | Epoch 12 | Val Acc: 94.10%
Round 0 | Epoch 13 | Val Acc: 94.30%
Round 0 | Epoch 14 | Val Acc: 95.60%
Round 0 | Epoch 15 | Val Acc: 95.30%
Round 0 | Epoch 16 | Val Acc: 95.60%
Round 0 | Epoch 17 | Val Acc: 96.00%
Round 0 | Epoch 18 | Val Acc: 96.20%
Round 0 | Epoch 19 | Val Acc: 96.10%
Round 0 | Epoch 20 | Val Acc: 96.10%
Round 0 | Epoch 21 | Val Acc: 95.70%
Round 0 | Epoch 22 | Val Acc: 95.60%
Round 0 | Epoch 23 | Val Acc: 95.90%
Round 0 | Epoch 24 | Val Acc: 96.00%
Round 0 | Epoch 25 | Val Acc: 95.70%
Round 0 | Epoch 26 | Val Acc: 96.50%
Roun

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Round 9 | Epoch 200 | Val Acc: 98.10%
Sparsity after pruning: 86.58%
Layer: features.0           | Active:      100 /      160 (62.50%)
Layer: features.5           | Active:     1142 /    10240 (11.15%)
Layer: features.10          | Active:     3482 /    40960 (8.50%)
Layer: classifier.1         | Active:     2995 /     8192 (36.56%)
Layer: classifier.4         | Active:      359 /      640 (56.09%)
------------------------------------------------------------
TOTAL ACTIVE PARAMETERS: 8,078 / 60,192
GLOBAL DENSITY: 13.42%
Active parameters after pruning: 8078
Weights reset to initialization.


active_parameters,█▆▅▄▃▃▂▂▁▁
best_val_acc,▅█▅▄▇██▇▁▁
round,▁▂▃▃▄▅▆▆▇█
round_0/train_loss,█▇▆▅▅▄▄▄▄▄▃▃▂▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
round_0/val_acc,▁▁▂▂▄▅▄▄▄▆▆▇▆▆▇▇▇█▇▇▇▇▇▇▇██████▇███▇▇███
round_1/train_loss,███▇▅▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁
round_1/val_acc,▁▃▃▃▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇▇███▇████▇██
round_2/train_loss,█▇█▇▆▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
round_2/val_acc,▁▂▅▆▆▆▆▆▇▆▇▆▇▆▇▆▇▇▇▆▇▇▇▇▇▇▇▇▇▇▇████▇▇▇█▇
round_3/train_loss,█▆▆▅▅▄▄▄▃▃▂▃▃▂▂▂▂▂▂▂▁▁▂▂▂▂▁▂▁▂▁▁▁▁▁▁▁▁▁▁
+14,...


Initial parameters: 60938

--- Round 0 ---
Round 0 | Epoch 1 | Val Acc: 89.00%
Round 0 | Epoch 2 | Val Acc: 92.10%
Round 0 | Epoch 3 | Val Acc: 92.70%
Round 0 | Epoch 4 | Val Acc: 92.50%
Round 0 | Epoch 5 | Val Acc: 93.10%
Round 0 | Epoch 6 | Val Acc: 93.10%
Round 0 | Epoch 7 | Val Acc: 93.10%
Round 0 | Epoch 8 | Val Acc: 93.00%
Round 0 | Epoch 9 | Val Acc: 92.90%
Round 0 | Epoch 10 | Val Acc: 92.90%
Round 0 | Epoch 11 | Val Acc: 94.00%
Round 0 | Epoch 12 | Val Acc: 94.70%
Round 0 | Epoch 13 | Val Acc: 94.40%
Round 0 | Epoch 14 | Val Acc: 94.10%
Round 0 | Epoch 15 | Val Acc: 94.90%
Round 0 | Epoch 16 | Val Acc: 95.00%
Round 0 | Epoch 17 | Val Acc: 95.50%
Round 0 | Epoch 18 | Val Acc: 95.70%
Round 0 | Epoch 19 | Val Acc: 95.60%
Round 0 | Epoch 20 | Val Acc: 95.30%
Round 0 | Epoch 21 | Val Acc: 95.70%
Round 0 | Epoch 22 | Val Acc: 95.30%
Round 0 | Epoch 23 | Val Acc: 96.00%
Round 0 | Epoch 24 | Val Acc: 96.20%
Round 0 | Epoch 25 | Val Acc: 96.20%
Round 0 | Epoch 26 | Val Acc: 96.40%
Roun

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Round 9 | Epoch 200 | Val Acc: 98.10%
Sparsity after pruning: 86.58%
Layer: features.0           | Active:      103 /      160 (64.38%)
Layer: features.5           | Active:     1149 /    10240 (11.22%)
Layer: features.10          | Active:     3478 /    40960 (8.49%)
Layer: classifier.1         | Active:     2995 /     8192 (36.56%)
Layer: classifier.4         | Active:      353 /      640 (55.16%)
------------------------------------------------------------
TOTAL ACTIVE PARAMETERS: 8,078 / 60,192
GLOBAL DENSITY: 13.42%
Active parameters after pruning: 8078
Weights reset to initialization.


active_parameters,█▆▅▄▃▃▂▂▁▁
best_val_acc,▄█▇▇▇██▇█▁
round,▁▂▃▃▄▅▆▆▇█
round_0/train_loss,██▇▆▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁
round_0/val_acc,▂▁▁▅▅▅▆▆▆▇▇▇▇▇█▇▇▇▇▇▇▇▇▇▇██▇▇██▇██▇▇▇█▇█
round_1/train_loss,▇███▆▄▄▃▃▃▃▂▃▂▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▂▂▁▁▁▁▁▁▁
round_1/val_acc,▁▂▄▅▆▆▆▆▇▆▆▇▇▆▇▇▇█▇▇▇▇▇▇▇▇▇█▇▇███▇▇█▇▇█▇
round_2/train_loss,█▇▆▆▆▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▂▁▁▁▁▁▁▁▁
round_2/val_acc,▁▅▅▅▆▆▆▆▇▆▆▇▆▆▇▆▇▇█▇▇█▇▇▇█▇█▇▇▇▇█▇▇▇▇▇▇▇
round_3/train_loss,█▇▆▅▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+14,...
